In [7]:
!pip install -q wandb accelerate

In [20]:
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_KEY")

base_path = '/kaggle/working'
project_name = 'weight-vs-loss-convergence-pinn'
project_root = os.path.join(base_path, project_name)

if not os.path.exists(project_root):
    os.chdir(base_path)
    !git clone https://github.com/nthday-jpg/weight-vs-loss-convergence-pinn.git
    print("Repository cloned successfully!")
else:
    os.chdir(project_root)
    !git pull
    print("Repository updated successfully!")
%cd {base_path}

# Ensure project root is in PYTHONPATH for script's imports
os.environ['PYTHONPATH'] = project_root

Already up to date.
Repository updated successfully!
/kaggle/working


In [21]:
# Generate data
save_dir = os.path.join(project_root, 'data')
!python -m burgers.data.gen_data \
        --save_dir {save_dir} \
        --nn 511\
        --steps 200 \
        --nu 0.01\
        --t_final 1.0

weight-vs-loss-convergence-pinn
Solving Burgers equation with 511 spatial points and 200 time steps...
Viscosity nu = 0.010000
Integration successful: True
Number of function evaluations: 13694
Data saved to /kaggle/working/weight-vs-loss-convergence-pinn/data/burgers.pt
Data saved to /kaggle/working/weight-vs-loss-convergence-pinn/data/burgers.npz

Done! Data shape: (201, 511)
Time range: [0.000, 1.000]
Space range: [-1.000, 1.000]
Solution range: [-0.999981, 0.999981]

Total time: 2.01 seconds


In [10]:
import json

config = {
    'layers': [2, 64, 1],
    'learning_rate': 1e-3,
    'num_epochs': 10000,
    'batch_size': 64,
    'l2_reg': 1e-6,
    'log_interval': 100
}

# Save config to JSON file
config_path = os.path.join(project_root, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)


In [32]:
!accelerate launch {project_root}/burgers/train.py \
    --config_file {config_path} \
    --data_file {save_dir/burgers.pt}

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `2`
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/bin/python3: can't open file '/kaggle/working/weight-vs-loss-convergence-pinn/burgers/{project_root}/burgers/train.py': [Errno 2] No such file or directory
/usr/bin/python3: can't open file '/kaggle/working/weight-vs-loss-convergence-pinn/burgers/{project_root}/burgers/train.py': [Errno 2] No such file or directory
E0226 16:24:33.908000 456 torch/distributed/elastic/multiprocessing/api.py:882] failed (exitcode: 2) local_rank: 0 (pid: 474) of binary: /usr/bin/python3
Traceback (most rece